# AP1 — Sinais e Sistemas
## Processamento de Sinais I

Notebook para execução dos procedimentos solicitados na Aula Prática 1.

O notebook:
- gera os sinais das questões 1 e 2;
- reproduz os sinais no próprio Google Colab;
- lê os arquivos `handel.wav`, `h_banheiro.wav` e `sinal_taca.wav`;
- gera os gráficos usados no relatório;
- compara espectros;
- ajusta frequências de amostragem quando necessário;
- realiza as convoluções da Questão 6;
- salva as figuras com os nomes usados no Overleaf.

## 0. Preparação do ambiente

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from scipy.io import wavfile
from scipy.signal import chirp, fftconvolve, resample_poly
from scipy.fft import rfft, rfftfreq
from IPython.display import Audio, display
from google.colab import files
from math import gcd
import os

plt.rcParams["figure.figsize"] = (12, 4)
plt.rcParams["axes.grid"] = True

def para_mono_float(x):
    # Converte áudio para mono e ponto flutuante normalizado.
    x = np.asarray(x)

    if x.ndim > 1:
        x = x.mean(axis=1)

    if np.issubdtype(x.dtype, np.integer):
        limite = max(abs(np.iinfo(x.dtype).min), np.iinfo(x.dtype).max)
        x = x.astype(np.float64) / limite
    else:
        x = x.astype(np.float64)

    return x

def normalizar(x):
    x = np.asarray(x, dtype=float)
    pico = np.max(np.abs(x))
    return x if pico == 0 else x / pico

def ajustar_fs(x, fs_original, fs_destino):
    # Reamostra x para fs_destino.
    if fs_original == fs_destino:
        return x

    divisor = gcd(int(fs_original), int(fs_destino))
    up = int(fs_destino) // divisor
    down = int(fs_original) // divisor

    return resample_poly(x, up, down)

def espectro(x, fs):
    # Retorna frequências e espectro de magnitude normalizado.
    x = np.asarray(x, dtype=float)
    X = np.abs(rfft(x))
    f = rfftfreq(len(x), 1/fs)

    if np.max(X) != 0:
        X = X / np.max(X)

    return f, X

print("Bibliotecas carregadas.")

### Upload dos arquivos de áudio

Selecione simultaneamente:

- `handel.wav`
- `h_banheiro.wav`
- `sinal_taca.wav`

In [ ]:
uploaded = files.upload()

arquivos_necessarios = [
    "handel.wav",
    "h_banheiro.wav",
    "sinal_taca.wav"
]

for nome in arquivos_necessarios:
    if os.path.exists(nome):
        print(f"[OK] {nome}")
    else:
        print(f"[FALTANDO] {nome}")

## 1. Questão 1 — Sinais cossenoidais

In [ ]:
fs = 44100
T = 5

t = np.arange(0, T, 1/fs)

frequencias = [500, 5000, 10000]

sinais_cossenoidais = {
    f: np.cos(2*np.pi*f*t)
    for f in frequencias
}

print(f"Frequência de amostragem: {fs} Hz")
print(f"Duração: {T} s")
print(f"Número de amostras: {len(t)}")

### Questão 1(a) — Gráficos no domínio do tempo

In [ ]:
# Os sinais possuem 5 s, mas o gráfico mostra os primeiros 5 ms
# para permitir visualizar as oscilações.

tempo_visualizacao = 0.005
N_vis = int(tempo_visualizacao * fs)

nomes_figuras = {
    500: "1A_500Hz.PNG",
    5000: "1A_5000Hz.PNG",
    10000: "1A_10000Hz.PNG"
}

for f in frequencias:
    x = sinais_cossenoidais[f]

    plt.figure(figsize=(12,4))
    plt.plot(t[:N_vis]*1000, x[:N_vis])
    plt.xlabel("Tempo (ms)")
    plt.ylabel("Amplitude")
    plt.title(f"Sinal cossenoidal — {f} Hz")
    plt.tight_layout()
    plt.savefig(nomes_figuras[f], dpi=300, bbox_inches="tight")
    plt.show()

    print(f"Figura salva: {nomes_figuras[f]}")

### Questão 1(b) — Reprodução dos sinais

In [ ]:
for f in frequencias:
    print(f"Reproduzindo sinal de {f} Hz")
    display(Audio(sinais_cossenoidais[f], rate=fs))

### Questão 1(c) — Observação

Ao aumentar a frequência, aumenta o número de oscilações no mesmo intervalo de tempo. Na reprodução, a percepção também muda de um tom relativamente mais grave para tons progressivamente mais agudos.

## 2. Questão 2 — Chirps

In [ ]:
f0 = 500
f1 = 10000
T_chirp = 5

t_chirp = np.arange(0, T_chirp, 1/fs)

chirp_linear = chirp(
    t_chirp,
    f0=f0,
    f1=f1,
    t1=T_chirp,
    method="linear"
)

chirp_quadratico = chirp(
    t_chirp,
    f0=f0,
    f1=f1,
    t1=T_chirp,
    method="quadratic"
)

chirp_log = chirp(
    t_chirp,
    f0=f0,
    f1=f1,
    t1=T_chirp,
    method="logarithmic"
)

chirps = {
    "Linear": chirp_linear,
    "Quadrático": chirp_quadratico,
    "Logarítmico": chirp_log
}

print("Chirps gerados.")

### Questão 2(a) — Gráficos no domínio do tempo

In [ ]:
# Mostra um pequeno trecho em torno de t = 2,5 s
# para que as oscilações possam ser distinguidas.

centro = 2.5
janela = 0.02

i0 = int((centro - janela/2) * fs)
i1 = int((centro + janela/2) * fs)

fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)

for ax, (nome, sinal) in zip(axes, chirps.items()):
    ax.plot(t_chirp[i0:i1], sinal[i0:i1])
    ax.set_ylabel("Amplitude")
    ax.set_title(f"Chirp {nome}")
    ax.grid(True)

axes[-1].set_xlabel("Tempo (s)")
plt.tight_layout()
plt.savefig("2_CHIRP.PNG", dpi=300, bbox_inches="tight")
plt.show()

print("Figura salva: 2_CHIRP.PNG")

### Questão 2(b) — Reprodução dos chirps

In [ ]:
for nome, sinal in chirps.items():
    print(f"Reproduzindo chirp {nome}")
    display(Audio(sinal, rate=fs))

### Questão 2(c) — Observação

Todos os chirps percorrem a faixa de 500 Hz a 10 kHz em 5 segundos, mas a evolução da frequência ao longo do tempo depende do método de varredura utilizado.

## 3. Questão 3 — Arquivo `handel.wav`

In [ ]:
fs_handel, handel_raw = wavfile.read("handel.wav")
handel = para_mono_float(handel_raw)

t_handel = np.arange(len(handel)) / fs_handel

print(f"Frequência de amostragem original: {fs_handel} Hz")
print(f"Número de amostras: {len(handel)}")
print(f"Duração: {len(handel)/fs_handel:.3f} s")

### Questão 3(a) — Sinal no domínio do tempo

In [ ]:
plt.figure(figsize=(12,4))
plt.plot(t_handel, handel)
plt.xlabel("Tempo (s)")
plt.ylabel("Amplitude")
plt.title("handel.wav — domínio do tempo")
plt.tight_layout()
plt.savefig("3A_handel_wav.PNG", dpi=300, bbox_inches="tight")
plt.show()

print("Figura salva: 3A_handel_wav.PNG")

### Questão 3(b) — Reprodução em fs, 2fs e 4fs

In [ ]:
print("Frequência original")
display(Audio(handel, rate=fs_handel))

print("Dobro da frequência de reprodução")
display(Audio(handel, rate=2*fs_handel))

print("Quatro vezes a frequência de reprodução")
display(Audio(handel, rate=4*fs_handel))

### Questão 3(c) — Comparação dos espectros

In [ ]:
f_h, X_h = espectro(handel, fs_handel)

espectros_cos = {}
for freq in frequencias:
    espectros_cos[freq] = espectro(sinais_cossenoidais[freq], fs)

espectros_chirp = {}
for nome, sinal in chirps.items():
    espectros_chirp[nome] = espectro(sinal, fs)

fig, axes = plt.subplots(3, 1, figsize=(12, 11))

axes[0].plot(f_h, X_h)
axes[0].set_xlim(0, min(20000, fs_handel/2))
axes[0].set_title("Espectro — handel.wav")
axes[0].set_ylabel("Magnitude normalizada")

for freq, (f_cos, X_cos) in espectros_cos.items():
    axes[1].plot(f_cos, X_cos, label=f"{freq} Hz")

axes[1].set_xlim(0, 12000)
axes[1].set_title("Espectros — sinais cossenoidais")
axes[1].set_ylabel("Magnitude normalizada")
axes[1].legend()

for nome, (f_c, X_c) in espectros_chirp.items():
    axes[2].plot(f_c, X_c, label=nome)

axes[2].set_xlim(0, 12000)
axes[2].set_title("Espectros — chirps")
axes[2].set_xlabel("Frequência (Hz)")
axes[2].set_ylabel("Magnitude normalizada")
axes[2].legend()

plt.tight_layout()
plt.savefig("3C_handel_wav.PNG", dpi=300, bbox_inches="tight")
plt.show()

print("Figura salva: 3C_handel_wav.PNG")

### Observação da Questão 3

Os cossenos apresentam energia concentrada em frequências específicas. Os chirps distribuem energia ao longo da faixa percorrida pela varredura. O sinal `handel.wav` possui conteúdo espectral mais complexo, pois é formado por diversas componentes de frequência.

## 4. Questão 4 — Procedimento para obter a resposta ao impulso

Uma forma experimental de determinar a resposta ao impulso de uma sala é:

1. posicionar uma fonte sonora e um microfone no ambiente;
2. emitir um sinal conhecido, como um impulso curto, sweep senoidal ou ruído;
3. registrar no microfone o sinal resultante;
4. considerar a sala aproximadamente como um sistema linear e invariante no tempo;
5. usar a relação

\[
y[n] = x[n] * h[n]
\]

para estimar \(h[n]\), normalmente por deconvolução entre o sinal conhecido e o sinal medido.

A medição pode ser afetada por ruído externo, posição da fonte, posição do microfone, resposta dos equipamentos e alterações nas condições acústicas do ambiente.

## 5. Questão 5 — `h_banheiro.wav` e `sinal_taca.wav`

In [ ]:
fs_banheiro, h_raw = wavfile.read("h_banheiro.wav")
fs_taca, taca_raw = wavfile.read("sinal_taca.wav")

h_banheiro = para_mono_float(h_raw)
taca = para_mono_float(taca_raw)

t_h = np.arange(len(h_banheiro)) / fs_banheiro
t_taca_original = np.arange(len(taca)) / fs_taca

print("h_banheiro.wav")
print(f"  fs = {fs_banheiro} Hz")
print(f"  duração = {len(h_banheiro)/fs_banheiro:.3f} s")

print("sinal_taca.wav")
print(f"  fs = {fs_taca} Hz")
print(f"  duração = {len(taca)/fs_taca:.3f} s")

### Questão 5(a) — Gráficos no domínio do tempo

In [ ]:
plt.figure(figsize=(12,4))
plt.plot(t_h, h_banheiro)
plt.xlabel("Tempo (s)")
plt.ylabel("Amplitude")
plt.title("Resposta ao impulso — h_banheiro.wav")
plt.tight_layout()
plt.savefig("5A.PNG", dpi=300, bbox_inches="tight")
plt.show()

plt.figure(figsize=(12,4))
plt.plot(t_taca_original, taca)
plt.xlabel("Tempo (s)")
plt.ylabel("Amplitude")
plt.title("Sinal da taça — sinal_taca.wav")
plt.tight_layout()
plt.savefig("5A_parte2.PNG", dpi=300, bbox_inches="tight")
plt.show()

print("Figuras salvas: 5A.PNG e 5A_parte2.PNG")

### Questão 5(b) — Reprodução dos sinais

In [ ]:
print("Resposta ao impulso do banheiro")
display(Audio(h_banheiro, rate=fs_banheiro))

print("Sinal da taça")
display(Audio(taca, rate=fs_taca))

### Questão 5(c) — Observação

O sinal da taça apresenta comportamento transitório, enquanto a resposta ao impulso do banheiro possui uma cauda temporal associada às reflexões e à reverberação do ambiente.

## 6. Questão 6 — Convolução com a resposta do banheiro

In [ ]:
# A convolução deve ser feita com sinais na mesma frequência de amostragem.
# A frequência de h_banheiro.wav é usada como referência.

taca_ajustada = ajustar_fs(taca, fs_taca, fs_banheiro)
handel_ajustado = ajustar_fs(handel, fs_handel, fs_banheiro)

print(f"fs de referência: {fs_banheiro} Hz")
print(f"Taça: {fs_taca} Hz -> {fs_banheiro} Hz")
print(f"Handel: {fs_handel} Hz -> {fs_banheiro} Hz")

### Questão 6(a) — Convolução e respostas no tempo

In [ ]:
y_taca = fftconvolve(taca_ajustada, h_banheiro, mode="full")
y_handel = fftconvolve(handel_ajustado, h_banheiro, mode="full")

y_taca_norm = normalizar(y_taca)
y_handel_norm = normalizar(y_handel)

t_taca = np.arange(len(taca_ajustada)) / fs_banheiro
t_y_taca = np.arange(len(y_taca_norm)) / fs_banheiro

t_handel_aj = np.arange(len(handel_ajustado)) / fs_banheiro
t_y_handel = np.arange(len(y_handel_norm)) / fs_banheiro

plt.figure(figsize=(12,4))
plt.plot(t_taca, taca_ajustada)
plt.xlabel("Tempo (s)")
plt.ylabel("Amplitude")
plt.title("Sinal da taça na entrada")
plt.tight_layout()
plt.savefig("6A.png", dpi=300, bbox_inches="tight")
plt.show()

plt.figure(figsize=(12,4))
plt.plot(t_h, h_banheiro)
plt.xlabel("Tempo (s)")
plt.ylabel("Amplitude")
plt.title("Resposta ao impulso do banheiro")
plt.tight_layout()
plt.savefig("6A_2.png", dpi=300, bbox_inches="tight")
plt.show()

plt.figure(figsize=(12,4))
plt.plot(t_y_taca, y_taca_norm)
plt.xlabel("Tempo (s)")
plt.ylabel("Amplitude normalizada")
plt.title("Taça no banheiro — convolução")
plt.tight_layout()
plt.savefig("6A_3.png", dpi=300, bbox_inches="tight")
plt.show()

plt.figure(figsize=(12,4))
plt.plot(t_y_handel, y_handel_norm)
plt.xlabel("Tempo (s)")
plt.ylabel("Amplitude normalizada")
plt.title("handel.wav no banheiro — convolução")
plt.tight_layout()
plt.savefig("6A_handel_convolucao.PNG", dpi=300, bbox_inches="tight")
plt.show()

print("Figuras da Questão 6 salvas.")

### Questão 6(b) — Reprodução das respostas

In [ ]:
print("Taça original/reamostrada")
display(Audio(normalizar(taca_ajustada), rate=fs_banheiro))

print("Taça após convolução com h_banheiro")
display(Audio(y_taca_norm, rate=fs_banheiro))

print("handel.wav original/reamostrado")
display(Audio(normalizar(handel_ajustado), rate=fs_banheiro))

print("handel.wav após convolução com h_banheiro")
display(Audio(y_handel_norm, rate=fs_banheiro))

### Salvar os sinais convoluídos

In [ ]:
wavfile.write(
    "taca_no_banheiro.wav",
    fs_banheiro,
    np.int16(y_taca_norm * 32767)
)

wavfile.write(
    "handel_no_banheiro.wav",
    fs_banheiro,
    np.int16(y_handel_norm * 32767)
)

print("Arquivos salvos:")
print("- taca_no_banheiro.wav")
print("- handel_no_banheiro.wav")

### Questão 6(c) — Observação

A convolução faz com que as características presentes na resposta ao impulso do banheiro sejam incorporadas aos sinais de entrada. Isso produz uma cauda reverberante e uma percepção sonora semelhante à reprodução em um ambiente com reflexões acústicas.

## 7. Conferência dos arquivos gerados

In [ ]:
arquivos_relatorio = [
    "1A_500Hz.PNG",
    "1A_5000Hz.PNG",
    "1A_10000Hz.PNG",
    "2_CHIRP.PNG",
    "3A_handel_wav.PNG",
    "3C_handel_wav.PNG",
    "5A.PNG",
    "5A_parte2.PNG",
    "6A.png",
    "6A_2.png",
    "6A_3.png",
    "6A_handel_convolucao.PNG",
    "taca_no_banheiro.wav",
    "handel_no_banheiro.wav"
]

for arquivo in arquivos_relatorio:
    status = "OK" if os.path.exists(arquivo) else "NÃO GERADO"
    print(f"{status:12} | {arquivo}")

## Finalização

Após executar todas as células, as figuras principais estarão com os mesmos nomes utilizados no relatório em LaTeX.

A imagem `6A_handel_convolucao.PNG` é adicional e pode ser incluída na Questão 6 caso se deseje mostrar graficamente também a convolução do `handel.wav`.